# Tutorial 05 — Multi-Turn Sessions

**Optional API key** — 2 cells perform live model calls and skip automatically when
`OPENAI_API_KEY` is not set. All structural cells run without it.

A "session" in eXo-brain is more than a single prompt/response pair. This tutorial shows:
- How to build a session-aware adapter that tracks conversation history across turns
- How the `RuntimeTimeline` threads correlation IDs through every event
- How `TenantQuotaManager` enforces per-tenant active-job limits across turns
- What a `QuotaDecision(allowed=False)` looks like when the limit is reached

eXo-brain's built-in `OpenAIAgentsRuntimeAdapter` (in `src/runtime/openai_agents_runtime.py`)
handles session lifecycle. For full conversation history tracking we use the delegating
wrapper pattern introduced in Tutorial 02 — the same `OpenAIAgentsSDKAdapter` class.

In [1]:
import pathlib
import sys, os

_repo = pathlib.Path(os.path.abspath(".."))
sys.path.insert(0, str(_repo))
_contracts_src = _repo / "packages" / "eXo_adapters" / "packages" / "exo-brain-core-contracts" / "src"
if _contracts_src.is_dir():
    sys.path.insert(0, str(_contracts_src))

try:
    from dotenv import load_dotenv
    load_dotenv("../.env", override=False)
except ImportError:
    pass

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")
HAS_API_KEY = bool(OPENAI_API_KEY)
print(f"API key present: {HAS_API_KEY}")

API key present: True


## Part 1 — Wire the session infrastructure

We wire a `RuntimeTimeline` and `TenantQuotaManager` alongside the adapter.
These are independent of the model provider and work across any adapter.

In [2]:
from src.observability.timeline import RuntimeTimeline
from src.tenancy.quotas import TenantQuotaManager, QuotaDecision
from src.observability.logging import StructuredLogger
from src.observability.metrics import RuntimeMetrics

# Timeline tracks ordered events across all turns of a session
timeline = RuntimeTimeline()
logger = StructuredLogger()
metrics = RuntimeMetrics()

# Quota manager: allow at most 2 concurrent active jobs per tenant
quota_manager = TenantQuotaManager(max_active_jobs_per_tenant=2, hard_enforcement=True)

print("timeline     :", type(timeline).__name__)
print("quota_manager:", type(quota_manager).__name__, "| max_active_jobs:", quota_manager.max_active_jobs)

timeline     : RuntimeTimeline
quota_manager: TenantQuotaManager | max_active_jobs: 2


## Part 2 — Build a session-aware adapter with history tracking

The delegating wrapper pattern (introduced in Tutorial 02) stores conversation history
in an in-memory dict keyed by `session_id`. This is the layer eXo-brain sits between:
the model sees the growing history, but the framework controls what enters the session.

We define a minimal version of the adapter that tracks history without requiring
an API key.

In [3]:
import asyncio
from importlib import import_module

try:
    import_module("nest_asyncio").apply()
except ModuleNotFoundError:
    pass

from typing import Any, AsyncIterator

class SessionAdapter:
    """Minimal session-aware adapter for multi-turn demonstration.
    Tracks conversation history per session without requiring a model provider.
    """

    def __init__(self) -> None:
        self._sessions: dict[str, dict[str, Any]] = {}

    async def start_session(
        self,
        session_id: str,
        tenant_id: str = "default",
        metadata: dict[str, Any] | None = None,
    ) -> None:
        self._sessions[session_id] = {
            "tenant_id": tenant_id,
            "metadata": metadata or {},
            "history": [],  # list of {"role": ..., "content": ...} dicts
        }

    def record_turn(
        self,
        session_id: str,
        user_message: str,
        assistant_reply: str,
    ) -> int:
        """Append a turn to history. Returns new history length."""
        history = self._sessions[session_id]["history"]
        history.append({"role": "user",      "content": user_message})
        history.append({"role": "assistant", "content": assistant_reply})
        return len(history)

# Create adapter and start a session
session_adapter = SessionAdapter()
session_id = "session-multiturn-demo"

async def start():
    await session_adapter.start_session(
        session_id=session_id,
        tenant_id="tenant-acme",
        metadata={"purpose": "multi-turn demo"},
    )

asyncio.run(start())

session_data = session_adapter._sessions[session_id]
print("Session keys :", list(session_data.keys()))
print("History length (before turns):", len(session_data["history"]))
print("Tenant ID    :", session_data["tenant_id"])

Session keys : ['tenant_id', 'metadata', 'history']
History length (before turns): 0
Tenant ID    : tenant-acme


## Part 3 — History grows with each turn

Each `record_turn` call appends a user + assistant pair to the session history.
The model (if used) would receive the full history on each subsequent turn,
allowing it to reference previous context.

In [4]:
# Simulate 3 conversation turns
turns = [
    ("What is the capital of France?",   "The capital of France is Paris."),
    ("And what about Germany?",           "The capital of Germany is Berlin."),
    ("Which has more letters in its name?", "Berlin has 6 letters; Paris has 5. Berlin has more."),
]

for i, (user_msg, assistant_reply) in enumerate(turns, 1):
    history_len = session_adapter.record_turn(session_id, user_msg, assistant_reply)
    print(f"Turn {i}: history length = {history_len}")
    print(f"  User     : {user_msg}")
    print(f"  Assistant: {assistant_reply}")
    print()

# Show full history structure
history = session_adapter._sessions[session_id]["history"]
print(f"Total history entries: {len(history)}")
print(f"(= {len(history) // 2} turns × 2 messages each)")

Turn 1: history length = 2
  User     : What is the capital of France?
  Assistant: The capital of France is Paris.

Turn 2: history length = 4
  User     : And what about Germany?
  Assistant: The capital of Germany is Berlin.

Turn 3: history length = 6
  User     : Which has more letters in its name?
  Assistant: Berlin has 6 letters; Paris has 5. Berlin has more.

Total history entries: 6
(= 3 turns × 2 messages each)


## Part 4 — Correlation IDs thread through the timeline

Each turn appends events to the `RuntimeTimeline` using a per-turn correlation ID.
`timeline.entries_for(correlation_id)` retrieves all events for that specific turn.
`timeline.all_entries()` gives the complete ordered trace across all turns.

In [5]:
# Record timeline events for each turn (mirrors what a production adapter would do)
for i in range(1, 4):
    corr = f"turn-{session_id}-{i:03d}"
    timeline.append(
        correlation_id=corr,
        event="session.turn_started",
        payload={"session_id": session_id, "turn": i, "tenant_id": "tenant-acme"},
    )
    timeline.append(
        correlation_id=corr,
        event="session.turn_completed",
        payload={"session_id": session_id, "turn": i, "status": "success"},
    )

# Inspect per-turn events
for i in range(1, 4):
    corr = f"turn-{session_id}-{i:03d}"
    entries = timeline.entries_for(corr)
    print(f"Turn {i} ({corr[:30]}...): {len(entries)} events")
    for e in entries:
        print(f"  {e.event}")

print(f"\nTotal timeline entries across all turns: {len(timeline.all_entries())}")
print("PASS — correlation IDs thread through the timeline correctly")

Turn 1 (turn-session-multiturn-demo-00...): 2 events
  session.turn_started
  session.turn_completed
Turn 2 (turn-session-multiturn-demo-00...): 2 events
  session.turn_started
  session.turn_completed
Turn 3 (turn-session-multiturn-demo-00...): 2 events
  session.turn_started
  session.turn_completed

Total timeline entries across all turns: 6
PASS — correlation IDs thread through the timeline correctly


## Part 5 — Quota enforcement: allowed and denied

`TenantQuotaManager.check_submission(tenant_id, active_jobs)` enforces the per-tenant
active job limit. It returns a `QuotaDecision` with `allowed`, `reason_code`, and `message`.

This same check runs before each background job submission — making it equally relevant
to multi-turn sessions that submit background work per turn.

In [6]:
TENANT = "tenant-acme"

# Under limit — allowed
decision_ok = quota_manager.check_submission(tenant_id=TENANT, active_jobs=0)
print("active_jobs=0 :", decision_ok)
assert decision_ok.allowed, "Should be allowed when under limit"

decision_ok2 = quota_manager.check_submission(tenant_id=TENANT, active_jobs=1)
print("active_jobs=1 :", decision_ok2)
assert decision_ok2.allowed, "Should be allowed at 1 (limit is 2)"

# At limit — hard enforcement blocks submission
decision_denied = quota_manager.check_submission(tenant_id=TENANT, active_jobs=2)
print("active_jobs=2 :", decision_denied)
assert not decision_denied.allowed, "Should be denied at limit"
assert decision_denied.reason_code in ("TENANT_QUOTA_EXCEEDED", "TENANT_QUOTA_SOFT_LIMIT")

print()
print("PASS — quota_manager enforces limits correctly")
print(f"Denied reason_code : {decision_denied.reason_code}")
print(f"Denied message     : {decision_denied.message}")

active_jobs=0 : QuotaDecision(allowed=True, reason_code='', message='')
active_jobs=1 : QuotaDecision(allowed=True, reason_code='', message='')
active_jobs=2 : QuotaDecision(allowed=False, reason_code='TENANT_QUOTA_EXCEEDED', message="Tenant 'tenant-acme' exceeded max active jobs quota.")

PASS — quota_manager enforces limits correctly
Denied reason_code : TENANT_QUOTA_EXCEEDED
Denied message     : Tenant 'tenant-acme' exceeded max active jobs quota.


## Part 6 — Live multi-turn conversation [REQUIRES API KEY]

This cell runs 3 real conversation turns with the OpenAI model on the same `session_id`.

**Enterprise proof rule:** cross-turn “memory” is only credible if you can see:

- the same `session_id` reused,
- a tool actually executed (tool intent/progress, not just “right” math),
- the third answer referencing the prior two capitals without asking for clarification.

**Skip this cell if you do not have `OPENAI_API_KEY` set.**

In [7]:
if not HAS_API_KEY:
    print("Skipping live turns — OPENAI_API_KEY not set.")
else:
    import uuid
    import json

    from src.core.orchestrator import Orchestrator
    from src.runtime.openai_agents_runtime import OpenAIAgentsRuntimeAdapter
    from src.tools.registry import ToolRegistry, ToolDescriptor
    from src.tools.executor import DeterministicToolExecutor
    from src.schemas.tool_io import RiskTier
    from src.policies.middleware import DeterministicFirstPolicyMiddleware
    from src.schemas.events import RuntimeEventType

    registry_live = ToolRegistry()
    policy_live = DeterministicFirstPolicyMiddleware()
    executor_live = DeterministicToolExecutor(registry=registry_live, policy=policy_live)

    def get_capital(country: str) -> str:
        """Returns the capital city of a country."""
        capitals = {"france": "Paris", "germany": "Berlin", "japan": "Tokyo"}
        return capitals.get(country.lower(), f"Unknown: {country}")

    _GET_CAPITAL_SCHEMA: dict[str, object] = {
        "type": "object",
        "properties": {"country": {"type": "string"}},
        "required": ["country"],
    }

    registry_live.register(
        ToolDescriptor(
            name="get_capital",
            handler=lambda country: {"capital": get_capital(str(country))},
            risk_tier=RiskTier.LOW,
            is_state_changing=False,
            description="Return capital city for a given country.",
            parameters_schema=_GET_CAPITAL_SCHEMA,
        )
    )

    live_session_id = "session-live-multiturn-05"
    live_sessions: dict[str, dict] = {}

    async def run_live_turns():
        live_sessions[live_session_id] = {"history": [], "tenant_id": "tenant-acme"}

        adapter_live = OpenAIAgentsRuntimeAdapter(
            provider_id="openai-gpt4o-mini",
            tool_registry=registry_live,
            tool_executor=executor_live,
        )
        await adapter_live.start_session(session_id=live_session_id, metadata={})

        orch = Orchestrator(runtime_adapter=adapter_live, policy_middleware=policy_live, tool_executor=executor_live)

        session_metadata = {
            "tenant_id": "tenant-acme",
            "agent_id": "nb-live-multiturn",
            "model": "gpt-4o-mini",
            "instructions": (
                "You are a concise assistant in a multi-turn session. "
                "Use get_capital when asked about a country's capital. "
                "Do not ask clarifying questions for the prompts in this demo; infer intent from prior turns. "
                "When comparing 'those two capitals', use the capitals from earlier turns."
            ),
        }

        prompts = [
            "What is the capital of France? Call get_capital(country='France') once.",
            "And what about Germany? Call get_capital(country='Germany') once.",
            "Which of those two capitals has more letters in its name? Answer in one sentence.",
        ]

        for i, prompt in enumerate(prompts, 1):
            print(f"\n--- Turn {i} ---")
            print(f"User: {prompt}")
            live_sessions[live_session_id]["history"].append({"role": "user", "content": prompt})

            reply_parts = []
            tools_completed: list[str] = []
            async for event in orch.run_turn(
                session_id=live_session_id,
                user_input=prompt,
                context={
                    "run_id": f"run-{i}",
                    "job_id": "job-live",
                    "task_id": "task-live",
                    "agent_id": "agent-live",
                    "session_metadata": dict(session_metadata),
                },
            ):
                if event.event_type == RuntimeEventType.TOOL_PROGRESS and isinstance(event.payload, dict):
                    if event.payload.get("state") == "completed":
                        tn = event.payload.get("tool_name")
                        if isinstance(tn, str) and tn:
                            tools_completed.append(tn)
                if event.event_type == RuntimeEventType.OUTPUT_DELTA:
                    text = str(event.payload.get("text", ""))
                    if text:
                        reply_parts.append(text)

            reply = "".join(reply_parts) or "(model response)"
            live_sessions[live_session_id]["history"].append({"role": "assistant", "content": reply})
            print(f"Assistant: {reply[:120]}")
            if i in (1, 2):
                print("Tools completed:", tools_completed or "none (unexpected)")
            print(f"History length: {len(live_sessions[live_session_id]['history'])}")

    asyncio.run(run_live_turns())


--- Turn 1 ---
User: What is the capital of France? Call get_capital(country='France') once.
Assistant: The capital of France is Paris.
Tools completed: none (unexpected)
History length: 2

--- Turn 2 ---
User: And what about Germany? Call get_capital(country='Germany') once.
Assistant: The capital of Germany is Berlin.
Tools completed: none (unexpected)
History length: 4

--- Turn 3 ---
User: Which of those two capitals has more letters in its name? Answer in one sentence.
Assistant: Between the two capitals, the one with more letters in its name is [insert longer capital name here].
History length: 6


## Summary

| Capability | Module | Key API |
|---|---|---|
| Session lifecycle | `src/runtime/openai_agents_runtime` | `OpenAIAgentsRuntimeAdapter.start_session()` |
| Cross-turn history | custom adapter pattern | `_sessions[session_id]["history"]` |
| Cross-turn correlation | `src/observability/timeline` | `timeline.append()`, `timeline.entries_for()` |
| Quota enforcement | `src/tenancy/quotas` | `quota_manager.check_submission()` |
| Quota denied | `src/tenancy/quotas` | `QuotaDecision(allowed=False, reason_code="TENANT_QUOTA_EXCEEDED")` |

**Key insight:** Session state (conversation history) lives in the adapter layer.
The `RuntimeTimeline` links every event back to its session via correlation ID.
Quota enforcement is stateless — the caller tracks `active_jobs` and the manager decides
allow/deny. The quota and timeline primitives are provider-independent; live model behaviour varies by adapter.

### Next steps
- **Tutorial 06** — Background workflows: long-running DAG jobs with retries and checkpointing
- **Tutorial 07** — Governance and anomaly detection: detect runaway tenants before they impact others

## Notebook navigation

| If you want… | Open |
|---|---|
| Previous / next in learning path | See `notebooks/README.md` index |
| Fast module smoke after a code change | `check_01` … `check_04` |
| Ingress or tool boundary proofs | `edge_01`, `edge_02` |
| Full governance lab (story + optional live) | `tutorial_08_governed_execution_sandbox.ipynb` |
| Evaluator time-boxed paths | `notebooks/EVALUATOR_GUIDE.md` |

**Regenerate notebooks:** edit this build script, then `python notebooks/build_tutorials.py` (do not hand-edit `.ipynb` JSON).